# Create a local voice profile from the consented video clip

This repository's approved sample is short, so this notebook uses **OpenVoice V2 tone-color conversion**; it does not fine-tune a Piper model. It uses only `00:56–01:28` from `voice_clone/src/sample_audio.mp4` because the earlier part contains a different speaker. The audio and speaker profile stay on this machine; the notebook does not upload them.

Run the cells in order from the repository root. The existing consent gate must pass before audio extraction or profile creation. The current consent record still has placeholder signed-date/document fields, so finish the signed consent record first; this notebook intentionally does not bypass or edit it. OpenVoice dependencies must already be installed in `tools/openvoice-venv` as described in `voice_clone/README.md`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "voice_clone").is_dir():
    ROOT = ROOT.parent
VIDEO = ROOT / "voice_clone" / "src" / "sample_audio.mp4"
VOICE_KEY = "yash"
SPEAKER_RANGE = "00:56-01:28"
WAV = ROOT / "voice_clone" / "work" / "audio" / f"{VOICE_KEY}.wav"
REPO_PYTHON = ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
OPENVOICE_PYTHON = ROOT / "tools" / "openvoice-venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
CLONE_SCRIPT = ROOT / "voice_clone" / "clone_speaker.py"
CONSENT_SCRIPT = ROOT / "voice_clone" / "consent.py"

assert VIDEO.is_file(), f"Source video not found: {VIDEO}"
assert REPO_PYTHON.is_file(), f"Project Python environment not found: {REPO_PYTHON}"
assert OPENVOICE_PYTHON.is_file(), f"OpenVoice environment not found: {OPENVOICE_PYTHON}"
print(f"Video: {VIDEO}\nClip: {SPEAKER_RANGE}\nOutput WAV: {WAV}")

## 1. Check written permission

This invokes the same gate as the production scripts. If it reports a missing date or signed document, stop here and complete `voice_clone/consent/consent.json` with the real signed record and place that document in the consent folder. Do not use an unsigned template as proof.

In [ ]:
sys.path.insert(0, str(ROOT / "voice_clone"))
from consent import require_consent

CONSENT = require_consent()
print(f"Consent gate passed for: {CONSENT['speaker_name']}")

## 2. Extract only the permitted speaker segment

The project script converts this segment to normalized, mono 22.05 kHz WAV and checks consent again. Review the video/clip boundaries before using a different range.

In [ ]:
subprocess.run(
    [str(REPO_PYTHON), str(ROOT / "voice_clone" / "get_audio.py"), "--file", str(VIDEO),
     "--ranges", SPEAKER_RANGE, "--name", VOICE_KEY],
    cwd=ROOT,
    check=True,
)
assert WAV.is_file(), f"Expected extracted WAV was not created: {WAV}"

## 3. Listen to and approve the extracted audio

Check that this is only the intended speaker, with clear speech and no music/interviewer. Stop rather than creating a profile if the segment contains another speaker or poor-quality audio.

In [ ]:
import wave
from IPython.display import Audio, display

with wave.open(str(WAV), "rb") as audio_file:
    seconds = audio_file.getnframes() / audio_file.getframerate()
    print(f"Duration: {seconds:.1f}s | Sample rate: {audio_file.getframerate()} Hz | Channels: {audio_file.getnchannels()}")
display(Audio(filename=str(WAV)))

## 4. Create the OpenVoice speaker profile

This extracts and averages short speaker-reference embeddings; it does not train new model weights. A clean 20–60 seconds is the intended range for this lightweight conversion route.

In [ ]:
subprocess.run(
    [str(OPENVOICE_PYTHON), "-W", "ignore", str(CLONE_SCRIPT), "--name", VOICE_KEY, "--wav", str(WAV)],
    cwd=ROOT,
    check=True,
)
PROFILE = ROOT / "voice_models" / "clone" / f"{VOICE_KEY}.target.pt"
assert PROFILE.is_file(), f"Speaker profile was not created: {PROFILE}"
print(f"Profile ready: {PROFILE}")

## 5. Connect it to the assistant and test

Set these entries in the project's local `.env` (do not commit that file), start `voice_clone/start_voice_service.ps1` in a separate PowerShell window, and restart the assistant:

```dotenv
VOICE_CONVERT_URL=http://127.0.0.1:8020
VOICE_CONVERT_SPEAKER=yash
```

Then run the next cell. It uses the repository's consent-gated acceptance check and requires the local converter service to be running.

In [ ]:
subprocess.run(
    [str(REPO_PYTHON), str(ROOT / "voice_clone" / "verify_clone.py"), "--speaker", VOICE_KEY],
    cwd=ROOT,
    check=True,
)